# 🎙️ DeepBrief.AI — Stage 1: Data Loading & Environment Setup

Loads the AMI corpus from HuggingFace, validates the environment, and selects representative
meeting samples for consistent use across all pipeline stages.

**Run cells top to bottom. Fix any ✗ errors before moving to Stage 2.**

> ℹ️ **Uses `streaming=False` (full cached download).** The dataset is cached on first run only.
> Change `HF_CACHE_DIR` in Cell 2 to point to a drive with enough free space (~29 GB).

## 📦 Cell 1 — Install Dependencies

Run once. Skip if already installed.

In [ ]:
# %pip install -r requirements.txt

## ⚙️ Cell 2 — Imports & Configuration

`NUM_SAMPLES` controls how many clips are extracted. `NUM_MEETINGS` controls how many
**distinct meetings** those clips are drawn from — spreading samples across meetings
gives more representative evaluation data and avoids all clips coming from one session.

In [1]:
import os
import io
import json
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# ─── HuggingFace cache — change D:/AI_Data to match your setup ────────────────
HF_CACHE_DIR = r"D:/AI_Data"
os.environ["HF_DATASETS_CACHE"] = HF_CACHE_DIR
os.environ["HF_HOME"]           = HF_CACHE_DIR
os.environ["TMPDIR"]            = HF_CACHE_DIR + "/tmp"
os.environ["TEMP"]              = HF_CACHE_DIR + "/tmp"
os.environ["TMP"]               = HF_CACHE_DIR + "/tmp"

# ─── Paths ────────────────────────────────────────────────────────────────────
SAMPLES_DIR      = Path("data/ami_samples")
SAMPLES_MANIFEST = Path("data/samples_manifest.json")

# ─── Sample selection ─────────────────────────────────────────────────────────
NUM_MEETINGS = 10   # distinct meetings to sample from
CLIPS_PER_MEETING = 2   # clips per meeting  →  total = NUM_MEETINGS * CLIPS_PER_MEETING

print(f"✓ Config loaded — targeting {NUM_MEETINGS} meetings × {CLIPS_PER_MEETING} clips = {NUM_MEETINGS * CLIPS_PER_MEETING} total samples")

✓ Config loaded — targeting 10 meetings × 2 clips = 20 total samples


## 🔍 Cell 3 — Environment Check

Checks your `.env` API key and all required packages. Fix any `✗` before continuing.

In [2]:
print("=" * 60)
print("DeepBrief.AI — Environment Check")
print("=" * 60)

issues = []

groq_key = os.getenv("GROQ_API_KEY")
if groq_key:
    print(f"  [✓] GROQ_API_KEY found ({groq_key[:8]}...)")
else:
    issues.append("GROQ_API_KEY not set in .env")
    print("  [✗] GROQ_API_KEY missing — add it to your .env file")

packages = {
    "groq": "groq", "datasets": "datasets", "transformers": "transformers",
    "streamlit": "streamlit", "rouge_score": "rouge-score",
    "jiwer": "jiwer", "torch": "torch", "soundfile": "soundfile",
}
for module, package in packages.items():
    try:
        __import__(module)
        print(f"  [✓] {package}")
    except ImportError:
        issues.append(f"Package not installed: {package}")
        print(f"  [✗] {package} — run: pip install {package}")

print()
if issues:
    print(f"  ⚠ {len(issues)} issue(s) found:")
    for issue in issues: print(f"     - {issue}")
else:
    print("  ✓ All checks passed. Ready to load data.")

DeepBrief.AI — Environment Check
  [✓] GROQ_API_KEY found (gsk_lwX8...)
  [✓] groq


c:\Users\sidsu\Desktop\Sem 4\AI\AT3\at3env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  [✓] datasets
  [✓] transformers
  [✓] streamlit
  [✓] rouge-score
  [✓] jiwer
  [✓] torch
  [✓] soundfile

  ✓ All checks passed. Ready to load data.


## 📥 Cell 4 — Load AMI Dataset

Downloads and caches the full AMI corpus (~29 GB on first run, instant on subsequent runs).
Using `split="train"` which contains 108,502 rows across many distinct meetings.

In [3]:
from datasets import load_dataset, Audio

print("=" * 60)
print("Loading AMI Corpus from HuggingFace...")
print(f"Dataset : edinburghcstr/ami  |  Config : ihm  |  Licence : CC BY 4.0")
print(f"Cache   : {HF_CACHE_DIR}")
print("⏳ First run downloads ~29 GB. Subsequent runs load from cache instantly.")
print("=" * 60)

dataset = load_dataset(
    "edinburghcstr/ami",
    "ihm",
    split="train",
    trust_remote_code=True,
)

print(f"\n  ✓ Dataset loaded — {len(dataset):,} rows")
print(f"  Columns: {dataset.column_names}")

Loading AMI Corpus from HuggingFace...
Dataset : edinburghcstr/ami  |  Config : ihm  |  Licence : CC BY 4.0
Cache   : D:/AI_Data
⏳ First run downloads ~29 GB. Subsequent runs load from cache instantly.

  ✓ Dataset loaded — 108,502 rows
  Columns: ['meeting_id', 'audio_id', 'text', 'audio', 'begin_time', 'end_time', 'microphone_id', 'speaker_id']


## 💾 Cell 5 — Select Samples Across Distinct Meetings

Selects `CLIPS_PER_MEETING` clips from each of `NUM_MEETINGS` distinct meetings,
ensuring samples are spread across different sessions rather than all from one meeting.

**Why this matters for WER evaluation:** If all samples come from one meeting (e.g. `EN2001a`),
the evaluation only reflects performance on one acoustic environment and one set of speakers.
Spreading across meetings gives a more honest and defensible accuracy figure in the report.

In [4]:
import soundfile as sf
import numpy as np

SAMPLES_DIR.mkdir(parents=True, exist_ok=True)
SAMPLES_MANIFEST.parent.mkdir(parents=True, exist_ok=True)

# Disable HF audio decoding — read raw bytes with soundfile directly
dataset = dataset.cast_column("audio", Audio(decode=False))

# Build a meeting_id → list of row indices map
print("Scanning dataset for distinct meetings...")
meeting_index: dict[str, list[int]] = {}
for i, row in enumerate(dataset):
    mid = row.get("meeting_id", "unknown")
    meeting_index.setdefault(mid, []).append(i)

all_meetings = list(meeting_index.keys())
print(f"  ✓ Found {len(all_meetings)} distinct meetings in train split")
print(f"  Selecting {NUM_MEETINGS} meetings X {CLIPS_PER_MEETING} clips\n")

# Pick the first NUM_MEETINGS meetings that have at least CLIPS_PER_MEETING rows
selected_meetings = [m for m in all_meetings if len(meeting_index[m]) >= CLIPS_PER_MEETING][:NUM_MEETINGS]

manifest = []
SAMPLES_DIR.mkdir(parents=True, exist_ok=True)

for meeting_id in selected_meetings:
    indices = meeting_index[meeting_id][:CLIPS_PER_MEETING]

    for idx in indices:
        sample    = dataset[idx]
        sample_id = f"{meeting_id}_clip{indices.index(idx):02d}"
        audio_info = sample["audio"]

        # Read audio bytes — avoids torchcodec/FFmpeg dependency
        if audio_info.get("bytes"):
            audio_array, sample_rate = sf.read(io.BytesIO(audio_info["bytes"]))
        else:
            filename = Path(audio_info["path"]).name
            matches  = list(Path(HF_CACHE_DIR).rglob(filename))
            if not matches:
                print(f"  [!] Could not find {filename} — skipping")
                continue
            audio_array, sample_rate = sf.read(str(matches[0]))

        audio_path = SAMPLES_DIR / f"{sample_id}.wav"
        sf.write(str(audio_path), audio_array, sample_rate)

        entry = {
            "sample_id":         sample_id,
            "dataset_index":     idx,
            "audio_path":        str(audio_path),
            "sampling_rate":     sample_rate,
            "duration_seconds":  round(len(audio_array) / sample_rate, 2),
            "speaker_id":        sample.get("speaker_id",  "unknown"),
            "meeting_id":        meeting_id,
            "begin_time":        sample.get("begin_time",  None),
            "end_time":          sample.get("end_time",    None),
            "ground_truth_text": sample.get("text",        ""),
        }
        manifest.append(entry)
        print(f"  ✓  {sample_id}  |  {entry['duration_seconds']}s  |  speaker: {entry['speaker_id']}")

with open(SAMPLES_MANIFEST, "w") as f:
    json.dump(manifest, f, indent=2)

print(f"\n  ✓ {len(manifest)} samples saved → {SAMPLES_DIR}/")
print(f"  ✓ Manifest written → {SAMPLES_MANIFEST}")

Scanning dataset for distinct meetings...
  ✓ Found 137 distinct meetings in train split
  Selecting 10 meetings X 2 clips

  ✓  EN2001a_clip00  |  4.21s  |  speaker: MEO069
  ✓  EN2001a_clip01  |  1.63s  |  speaker: MEE068
  ✓  EN2001b_clip00  |  4.21s  |  speaker: MEO069
  ✓  EN2001b_clip01  |  1.77s  |  speaker: FEO065
  ✓  EN2001d_clip00  |  0.95s  |  speaker: MEO069
  ✓  EN2001d_clip01  |  2.74s  |  speaker: MEE068
  ✓  EN2001e_clip00  |  0.88s  |  speaker: MEO069
  ✓  EN2001e_clip01  |  3.4s  |  speaker: MEO069
  ✓  EN2003a_clip00  |  4.26s  |  speaker: MEE075
  ✓  EN2003a_clip01  |  0.29s  |  speaker: MEO074
  ✓  EN2004a_clip00  |  3.38s  |  speaker: FEE078
  ✓  EN2004a_clip01  |  2.09s  |  speaker: FEE078
  ✓  EN2005a_clip00  |  2.24s  |  speaker: FEO084
  ✓  EN2005a_clip01  |  0.84s  |  speaker: FEE085
  ✓  EN2006a_clip00  |  4.49s  |  speaker: MEE089
  ✓  EN2006a_clip01  |  0.24s  |  speaker: FEE088
  ✓  EN2006b_clip00  |  4.76s  |  speaker: FEE087
  ✓  EN2006b_clip01  |  1.1

## 📋 Cell 6 — Sample Summary

In [7]:
print("=" * 60)
print("Selected Sample Summary")
print("=" * 60)

total_duration = sum(s["duration_seconds"] for s in manifest)
meetings_covered = len(set(s["meeting_id"] for s in manifest))

print(f"  Samples          : {len(manifest)}")
print(f"  Meetings covered : {meetings_covered}")
print(f"  Total duration   : {total_duration:.1f}s ({total_duration/60:.1f} min)")
print()

for s in manifest:
    print(f"  {s['sample_id']}  |  {s['duration_seconds']}s  |  {s['speaker_id']}  |  {s['meeting_id']}")
    if s["ground_truth_text"]:
        preview = s["ground_truth_text"][:80].replace("\n", " ")
        print(f'    GT: "{preview}"...')

print()
print("✅ Stage 1 complete. Proceed to stage2.ipynb")

Selected Sample Summary
  Samples          : 20
  Meetings covered : 10
  Total duration   : 46.4s (0.8 min)

  EN2001a_clip00  |  4.21s  |  MEO069  |  EN2001a
    GT: "IF YOU IF YOU S. S. H. AND THEY HAVE THIS BIG WARNING ABOUT DOING NOTHING AT ALL"...
  EN2001a_clip01  |  1.63s  |  MEE068  |  EN2001a
    GT: "I'VE GOTTEN MM HARDLY ANY"...
  EN2001b_clip00  |  4.21s  |  MEO069  |  EN2001b
    GT: "OH THIS IS HOW MUCH TIME YOU SPEND JUST GETTING THE RIGHT SOFTWARE GOING"...
  EN2001b_clip01  |  1.77s  |  FEO065  |  EN2001b
    GT: "YEAH YEAH PROBABLY"...
  EN2001d_clip00  |  0.95s  |  MEO069  |  EN2001d
    GT: "NO NO N"...
  EN2001d_clip01  |  2.74s  |  MEE068  |  EN2001d
    GT: "IT'S READY TO GET STARTED"...
  EN2001e_clip00  |  0.88s  |  MEO069  |  EN2001e
    GT: "YEAH OKAY COOL"...
  EN2001e_clip01  |  3.4s  |  MEO069  |  EN2001e
    GT: "BUT IT'S PROBABLY NOT AND UH NOT A SIMPLE THING"...
  EN2003a_clip00  |  4.26s  |  MEE075  |  EN2003a
    GT: "'CAUSE I MEAN WOULD SIX HOURS A 